# Random Forest Classification

Ensemble method that builds multiple decision trees and aggregates their predictions (bagging + feature randomness).

## Topics Covered
1. Random Forest Algorithm Intuition
2. Training and Evaluation
3. Feature Importance
4. Hyperparameter Tuning
5. Out-of-Bag (OOB) Error
6. Comparison with single Decision Tree

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, learning_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Explore Data

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target

df = pd.DataFrame(X, columns=wine.feature_names)
df['target'] = y
print(f"Shape: {df.shape}")
print(f"Classes: {np.unique(y)} (counts: {np.bincount(y)})")
df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 2. Decision Tree vs Random Forest

In [ ]:
# Single Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_acc = dt.score(X_test, y_test)
dt_cv = cross_val_score(dt, X, y, cv=10).mean()

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, oob_score=True)
rf.fit(X_train, y_train)
rf_acc = rf.score(X_test, y_test)
rf_cv = cross_val_score(rf, X, y, cv=10).mean()

print(f"{'Model':<20} {'Test Acc':>10} {'CV Acc':>10}")
print("-" * 42)
print(f"{'Decision Tree':<20} {dt_acc:>10.4f} {dt_cv:>10.4f}")
print(f"{'Random Forest':<20} {rf_acc:>10.4f} {rf_cv:>10.4f}")
print(f"\nOOB Score: {rf.oob_score_:.4f}")

In [ ]:
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=wine.target_names))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=wine.target_names)
disp.plot(cmap='Blues')
plt.title('Random Forest — Confusion Matrix')
plt.show()

## 3. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=wine.feature_names)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 8))
importances.plot(kind='barh', color='steelblue')
plt.xlabel('Feature Importance (Gini)')
plt.title('Random Forest Feature Importance — Wine Dataset')
plt.tight_layout()
plt.show()

print("\nTop 5 Features:")
print(importances.tail(5))

## 4. Effect of Number of Trees

In [ ]:
n_trees = [1, 5, 10, 25, 50, 100, 200, 500]
oob_scores = []
test_scores = []

for n in n_trees:
    rf_temp = RandomForestClassifier(
        n_estimators=n, random_state=42, oob_score=True
    )
    rf_temp.fit(X_train, y_train)
    oob_scores.append(rf_temp.oob_score_)
    test_scores.append(rf_temp.score(X_test, y_test))

plt.figure(figsize=(10, 5))
plt.plot(n_trees, oob_scores, 'o-', label='OOB Score', color='steelblue')
plt.plot(n_trees, test_scores, 's-', label='Test Score', color='coral')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Random Forest: Effect of n_estimators')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy',
    n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")
print(f"Test Score: {grid_search.score(X_test, y_test):.4f}")

## 6. Learning Curve

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X, y, train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='accuracy', n_jobs=-1
)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Training Score', color='steelblue')
plt.fill_between(train_sizes,
                 train_scores.mean(axis=1) - train_scores.std(axis=1),
                 train_scores.mean(axis=1) + train_scores.std(axis=1),
                 alpha=0.1, color='steelblue')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', label='Validation Score', color='coral')
plt.fill_between(train_sizes,
                 val_scores.mean(axis=1) - val_scores.std(axis=1),
                 val_scores.mean(axis=1) + val_scores.std(axis=1),
                 alpha=0.1, color='coral')
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.title('Random Forest — Learning Curve')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Visualize a Single Tree from the Forest

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(rf.estimators_[0], feature_names=wine.feature_names,
          class_names=wine.target_names, filled=True,
          rounded=True, max_depth=3, ax=ax, fontsize=9)
plt.title('Sample Tree from Random Forest (max_depth=3 for visibility)')
plt.tight_layout()
plt.show()

## Key Takeaways

| Aspect | Details |
|--------|---------|  
| **Ensemble Type** | Bagging (Bootstrap Aggregating) + Feature randomness |
| **Variance Reduction** | Averages many deep trees → low variance |
| **Feature Scaling** | NOT required (tree-based) |
| **Key Hyperparameters** | n_estimators, max_depth, min_samples_split, max_features |
| **OOB Score** | Free validation using unsampled data |
| **Pros** | Handles non-linearity, feature importance, robust to outliers |
| **Cons** | Slower than single tree, less interpretable, memory-heavy |